# Extract Binance Spot Monthly Kline ZIP Files

This notebook only does extraction:
1. Finds all monthly ZIP files for a symbol/interval.
2. Extracts CSV content from ZIPs.
3. Adds spot kline headers to each extracted CSV.

It does **not** combine files. Use the separate combine notebook for that.


In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import io
import re
from zipfile import ZipFile

# -----------------------------
# Configuration
# -----------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "downloads").exists() and (PROJECT_ROOT.parent / "downloads").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SYMBOL = "BTCUSDT"
INTERVAL = "1s"
SOURCE_DIR = PROJECT_ROOT / "downloads" / "data" / "spot" / "monthly" / "klines" / SYMBOL / INTERVAL
OUTPUT_DIR = PROJECT_ROOT / "downloads" / "test1"
OVERWRITE_EXISTING = True

KLINE_COLUMNS = ['Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close time', 'Quote asset volume', 'Number of trades', 'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore']

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dir  : {SOURCE_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")


In [ ]:
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")

zip_pattern = re.compile(
    rf"^{re.escape(SYMBOL)}-{re.escape(INTERVAL)}-(\d{{4}}-\d{{2}})\.zip$",
    flags=re.IGNORECASE,
)

selected_zips = []
for zip_path in SOURCE_DIR.iterdir():
    if not zip_path.is_file() or zip_path.suffix.lower() != ".zip":
        continue
    match = zip_pattern.match(zip_path.name)
    if not match:
        continue
    month_text = match.group(1)
    month_dt = datetime.strptime(month_text, "%Y-%m")
    selected_zips.append((month_dt, month_text, zip_path))

selected_zips.sort(key=lambda x: x[0])

if not selected_zips:
    raise ValueError(f"No ZIP files found in expected format under {SOURCE_DIR}")

print(f"Selected ZIP files: {len(selected_zips)}")
for i, (_, month_text, zip_path) in enumerate(selected_zips, start=1):
    if i <= 10 or i > len(selected_zips) - 3:
        print(f"  {month_text}: {zip_path.name}")
    elif i == 11:
        print("  ...")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

extracted_csv_paths = []

for _, month_text, zip_path in selected_zips:
    with ZipFile(zip_path, "r") as zf:
        csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if not csv_members:
            raise ValueError(f"No CSV file found in ZIP: {zip_path}")

        if len(csv_members) > 1:
            print(
                f"Warning: {zip_path.name} contains {len(csv_members)} CSV files. "
                "All files will be processed."
            )

        for idx, member in enumerate(csv_members, start=1):
            suffix = "" if len(csv_members) == 1 else f"-part{idx}"
            output_name = f"{SYMBOL}-{INTERVAL}-{month_text}{suffix}.csv"
            output_csv_path = OUTPUT_DIR / output_name

            if output_csv_path.exists() and not OVERWRITE_EXISTING:
                print(f"Skipping existing file (OVERWRITE_EXISTING=False): {output_csv_path}")
                extracted_csv_paths.append(output_csv_path)
                continue

            with zf.open(member, "r") as src_bin, output_csv_path.open(
                "w", newline="", encoding="utf-8"
            ) as dst:
                reader = csv.reader(io.TextIOWrapper(src_bin, encoding="utf-8"))
                writer = csv.writer(dst)

                writer.writerow(KLINE_COLUMNS)

                first_row = next(reader, None)
                if first_row is not None:
                    # Most Binance files do not include a header. If one exists, do not duplicate it.
                    if first_row != KLINE_COLUMNS:
                        writer.writerow(first_row)
                    for row in reader:
                        writer.writerow(row)

            extracted_csv_paths.append(output_csv_path)
            print(f"Wrote: {output_csv_path}")

print(f"Finished extracting {len(extracted_csv_paths)} CSV file(s) to {OUTPUT_DIR}")


In [ ]:
print("Preview extracted files (up to 10):")
for i, path in enumerate(extracted_csv_paths[:10], start=1):
    print(f"  {i}. {path.name}")
if len(extracted_csv_paths) > 10:
    print(f"  ... and {len(extracted_csv_paths) - 10} more")
